# 21700 NMC (~5 Ah) Battery Cycling — Simulation + Degradation Analysis

This notebook simulates an **EV-style 21700 NMC cell (~5 Ah, 4.2 V max)** and produces an end-to-end portfolio workflow:

**Simulate**
- Charge/discharge cycling
- Voltage vs time
- SOC estimation (coulomb count + OCV correction during rest)
- Temperature trend (lumped thermal model)

**Analyze**
- Capacity fade trend
- Efficiency loss (coulombic + energy)
- Degradation curves

We run **two conditions**:
1. **Baseline**: 25°C, 1C charge/discharge
2. **Stress**: 35°C, 2C charge/discharge (faster aging)

All charts + CSV exports are written to `outputs/`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

outdir = Path("outputs")
outdir.mkdir(exist_ok=True)

np.random.seed(7)


## 1) Simple cell model

In [ ]:
def ocv_from_soc(soc: np.ndarray) -> np.ndarray:
    """Smooth OCV curve ~3.0–4.2V (synthetic but plausible)."""
    soc = np.clip(soc, 0, 1)
    return 3.0 + 1.2*soc - 0.1*np.sin(2*np.pi*soc) - 0.05*np.sin(6*np.pi*soc)

def degradation_laws(n_cycles: int,
                     Q0_Ah: float,
                     R0_ohm: float,
                     cap_fade_per_cycle: float,
                     R_growth_per_cycle: float,
                     seed: int = 1):
    """Return per-cycle capacity and resistance arrays."""
    rng = np.random.default_rng(seed)
    cap_noise = rng.normal(0, 0.00015, size=n_cycles)
    R_noise = rng.normal(0, 0.00025, size=n_cycles)

    Q = np.empty(n_cycles)
    R = np.empty(n_cycles)
    Q[0] = Q0_Ah
    R[0] = R0_ohm

    for k in range(1, n_cycles):
        Q[k] = Q[k-1] * (1 - (cap_fade_per_cycle + cap_noise[k]))
        R[k] = R[k-1] * (1 + (R_growth_per_cycle + R_noise[k]))
    return Q, R

def simulate_cycle(
    cycle_idx: int,
    Q_nom_Ah: float,
    R0_ohm: float,
    dt_s: float,
    I_charge_A: float,    # negative = charging
    I_discharge_A: float, # positive = discharging
    soc_start: float = 0.1,
    soc_end_charge: float = 0.9,
    soc_end_discharge: float = 0.1,
    T_amb_C: float = 25.0,
    C_th_J_per_K: float = 180.0,
    R_th_K_per_W: float = 2.0,
    eta_coulombic: float = 0.995,
    vpol_tau_s: float = 80.0,
    vpol_gain: float = 0.025,
):
    """One CC charge + rest + CC discharge cycle."""
    Q_As = Q_nom_Ah * 3600.0

    rows = []
    t = 0.0
    soc = soc_start
    T = T_amb_C
    vpol = 0.0

    def step(I):
        nonlocal t, soc, T, vpol
        # SOC update (charge efficiency applied only on charge)
        if I < 0:
            d_soc = (-I) * dt_s / Q_As * eta_coulombic
        else:
            d_soc = (-I) * dt_s / Q_As
        soc = float(np.clip(soc + d_soc, 0, 1))

        # polarization lag
        vpol += (-(vpol) + vpol_gain * I) * (dt_s / vpol_tau_s)

        ocv = float(ocv_from_soc(np.array([soc]))[0])
        V = ocv + I * R0_ohm + vpol

        # thermal: I^2 R heating, convective loss via R_th
        P = (I**2) * R0_ohm
        T += (P - (T - T_amb_C)/R_th_K_per_W) * (dt_s / C_th_J_per_K)

        rows.append([cycle_idx, t, I, soc, ocv, vpol, V, T, P])
        t += dt_s

    # Charge (CC only for simplicity)
    while soc < soc_end_charge:
        step(I_charge_A)

    # Rest
    for _ in range(int(60/dt_s)):
        step(0.0)

    # Discharge (CC)
    while soc > soc_end_discharge:
        step(I_discharge_A)

    return pd.DataFrame(rows, columns=["cycle","t_s","I_A","soc_true","ocv_V","vpol_V","V_V","T_C","P_W"])


## 2) Run two conditions: baseline vs stress

In [ ]:
n_cycles = 120
dt_s = 2.0  # timestep

# 21700 NMC ~5Ah
Q0_Ah = 5.0
R0_ohm = 0.020  # plausible ballpark for a high-power 21700 (synthetic)

configs = {
    "baseline_25C_1C": dict(
        T_amb_C=25.0,
        I_charge_A=-5.0,      # 1C
        I_discharge_A=5.0,    # 1C
        cap_fade_per_cycle=0.00035,  # slower aging
        R_growth_per_cycle=0.0010,
        eta_coulombic_start=0.9965,
        seed=10,
    ),
    "stress_35C_2C": dict(
        T_amb_C=35.0,
        I_charge_A=-10.0,     # 2C
        I_discharge_A=10.0,   # 2C
        cap_fade_per_cycle=0.00075,  # faster aging
        R_growth_per_cycle=0.0018,
        eta_coulombic_start=0.9955,
        seed=20,
    )
}

all_data = {}
all_kpis = {}

for name, cfg in configs.items():
    Q_Ah, R_ohm = degradation_laws(
        n_cycles=n_cycles,
        Q0_Ah=Q0_Ah,
        R0_ohm=R0_ohm,
        cap_fade_per_cycle=cfg["cap_fade_per_cycle"],
        R_growth_per_cycle=cfg["R_growth_per_cycle"],
        seed=cfg["seed"]
    )

    dfs = []
    for k in range(n_cycles):
        dfk = simulate_cycle(
            cycle_idx=k,
            Q_nom_Ah=float(Q_Ah[k]),
            R0_ohm=float(R_ohm[k]),
            dt_s=dt_s,
            I_charge_A=cfg["I_charge_A"],
            I_discharge_A=cfg["I_discharge_A"],
            T_amb_C=cfg["T_amb_C"],
            eta_coulombic=cfg["eta_coulombic_start"] - 0.0000015*k,
        )
        dfk["condition"] = name
        dfs.append(dfk)

    data = pd.concat(dfs, ignore_index=True)

    # SOC estimation
    data["I_meas_A"] = data["I_A"] + np.random.normal(0, 0.05, size=len(data))
    soc_est_all = np.empty(len(data), dtype=float)

    grid = np.linspace(0,1,201)
    Vgrid = ocv_from_soc(grid)

    for k in range(n_cycles):
        idx = data.index[data["cycle"]==k].to_numpy()
        I = data.loc[idx, "I_meas_A"].to_numpy()
        V = data.loc[idx, "V_V"].to_numpy()
        Itrue = data.loc[idx, "I_A"].to_numpy()
        Q_As = float(Q_Ah[k]) * 3600.0

        soc = 0.1 + np.random.normal(0, 0.006)
        est = np.empty(len(idx), dtype=float)

        for i in range(len(idx)):
            soc += (-I[i]) * dt_s / Q_As
            soc = float(np.clip(soc, 0, 1))
            if abs(Itrue[i]) < 0.05:
                soc_ocv = float(grid[np.argmin((Vgrid - V[i])**2)])
                soc = 0.96*soc + 0.04*soc_ocv
            est[i] = soc

        soc_est_all[idx] = est

    data["soc_est"] = soc_est_all

    # KPIs
    kpis = []
    dt_h = dt_s/3600.0
    for k in range(n_cycles):
        d = data[data["cycle"]==k]
        charge = d[d["I_A"] < -0.1]
        discharge = d[d["I_A"] > 0.1]

        Q_charge_Ah = float((-charge["I_A"]).sum() * dt_h)
        Q_dis_Ah = float((discharge["I_A"]).sum() * dt_h)

        E_charge_Wh = float(((-charge["I_A"]) * charge["V_V"]).sum() * dt_h)
        E_dis_Wh = float(((discharge["I_A"]) * discharge["V_V"]).sum() * dt_h)

        ce = Q_dis_Ah / Q_charge_Ah if Q_charge_Ah > 0 else np.nan
        ee = E_dis_Wh / E_charge_Wh if E_charge_Wh > 0 else np.nan

        Tmax = float(d["T_C"].max())
        Vmin = float(d["V_V"].min())
        Vmax = float(d["V_V"].max())

        kpis.append([k, Q_charge_Ah, Q_dis_Ah, ce, ee, Tmax, Vmin, Vmax, float(Q_Ah[k]), float(R_ohm[k])])

    kpis = pd.DataFrame(kpis, columns=[
        "cycle","Q_charge_Ah","Q_dis_Ah","coul_eff","energy_eff",
        "Tmax_C","Vmin_V","Vmax_V","Q_model_Ah","R_model_ohm"
    ])
    kpis["condition"] = name

    all_data[name] = data
    all_kpis[name] = kpis

# quick peek
all_kpis["baseline_25C_1C"].head()


## 3) Plots (comparison)

In [ ]:
def savefig(name: str):
    plt.tight_layout()
    plt.savefig(outdir / name, dpi=200)
    plt.close()

baseline = "baseline_25C_1C"
stress = "stress_35C_2C"

# Voltage vs time (example cycle)
cycle_show = 60
plt.figure(figsize=(9,4))
for cond in [baseline, stress]:
    d = all_data[cond][all_data[cond]["cycle"]==cycle_show]
    plt.plot(d["t_s"]/60, d["V_V"], label=cond)
plt.xlabel("Time (min)")
plt.ylabel("Voltage (V)")
plt.title(f"Voltage vs time — cycle {cycle_show}")
plt.legend()
savefig("compare_voltage_cycle60.png")

# SOC estimation (example cycle)
plt.figure(figsize=(9,4))
d = all_data[baseline][all_data[baseline]["cycle"]==cycle_show]
plt.plot(d["t_s"]/60, d["soc_true"], label="SOC true (baseline)")
plt.plot(d["t_s"]/60, d["soc_est"], label="SOC est (baseline)", alpha=0.85)
d2 = all_data[stress][all_data[stress]["cycle"]==cycle_show]
plt.plot(d2["t_s"]/60, d2["soc_est"], label="SOC est (stress)", alpha=0.85)
plt.xlabel("Time (min)")
plt.ylabel("SOC")
plt.title(f"SOC estimation — cycle {cycle_show}")
plt.legend()
savefig("compare_soc_est_cycle60.png")

# Temperature vs time (example cycle)
plt.figure(figsize=(9,4))
for cond in [baseline, stress]:
    d = all_data[cond][all_data[cond]["cycle"]==cycle_show]
    plt.plot(d["t_s"]/60, d["T_C"], label=cond)
plt.xlabel("Time (min)")
plt.ylabel("Temperature (C)")
plt.title(f"Temperature vs time — cycle {cycle_show}")
plt.legend()
savefig("compare_temperature_cycle60.png")

# Capacity fade
plt.figure(figsize=(8,4))
for cond in [baseline, stress]:
    kpis = all_kpis[cond]
    plt.plot(kpis["cycle"], kpis["Q_dis_Ah"], label=cond)
plt.xlabel("Cycle")
plt.ylabel("Discharge capacity (Ah)")
plt.title("Capacity fade trend (comparison)")
plt.legend()
savefig("compare_capacity_fade.png")

# Capacity retention
plt.figure(figsize=(8,4))
for cond in [baseline, stress]:
    kpis = all_kpis[cond]
    ret = kpis["Q_dis_Ah"]/kpis.loc[0,"Q_dis_Ah"]
    plt.plot(kpis["cycle"], ret, label=cond)
plt.xlabel("Cycle")
plt.ylabel("Capacity retention")
plt.title("Degradation curve — capacity retention")
plt.ylim(0.85, 1.02)
plt.legend()
savefig("compare_capacity_retention.png")

# Efficiency
plt.figure(figsize=(8,4))
for cond in [baseline, stress]:
    kpis = all_kpis[cond]
    plt.plot(kpis["cycle"], kpis["energy_eff"], label=f"Energy eff ({cond})")
plt.xlabel("Cycle")
plt.ylabel("Energy efficiency")
plt.title("Energy efficiency loss (comparison)")
plt.ylim(0.85, 1.01)
plt.legend()
savefig("compare_energy_efficiency.png")

plt.figure(figsize=(8,4))
for cond in [baseline, stress]:
    kpis = all_kpis[cond]
    plt.plot(kpis["cycle"], kpis["coul_eff"], label=f"Coulombic eff ({cond})")
plt.xlabel("Cycle")
plt.ylabel("Coulombic efficiency")
plt.title("Coulombic efficiency loss (comparison)")
plt.ylim(0.90, 1.01)
plt.legend()
savefig("compare_coulombic_efficiency.png")

# Resistance growth
plt.figure(figsize=(8,4))
for cond in [baseline, stress]:
    kpis = all_kpis[cond]
    plt.plot(kpis["cycle"], kpis["R_model_ohm"], label=cond)
plt.xlabel("Cycle")
plt.ylabel("Resistance (ohm)")
plt.title("Resistance growth trend (comparison)")
plt.legend()
savefig("compare_resistance_growth.png")


## 4) Export CSVs

In [ ]:
# Save per-condition exports
for cond, df in all_data.items():
    df.to_csv(outdir / f"simulated_timeseries_{cond}.csv", index=False)

for cond, kpi in all_kpis.items():
    kpi.to_csv(outdir / f"cycle_kpis_{cond}.csv", index=False)

print("Wrote outputs for:", list(all_data.keys()))
